In [ ]:
import json
import os
import glob
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Make the project root importable so `from src.bots import ...` works
sys.path.insert(0, str(Path.cwd().parent))

## Load data files & Mouse trajectory preview

In [ ]:
from src.plotting import plot_trajectory
from src.data import find_red_eclipse_files, load_red_eclipse_mouse

game_files = find_red_eclipse_files()

for file_path in game_files[:5]:
    meta, mouse = load_red_eclipse_mouse(file_path)
    if mouse.empty:
        continue
    plot_trajectory(mouse, title=f"userId={meta['userId']}, gameId={meta['gameId']}")
    print(f"\nuserId={meta['userId']}, gameId={meta['gameId']}, events={len(mouse)}")


## Feature extraction


In [ ]:
from src.features import extract_features
from src.data import load_red_eclipse_mouse

re_rows = []
re_human_traces = []

for file_path in game_files:
    re_meta, re_mouse = load_red_eclipse_mouse(file_path)
    features = extract_features(re_mouse)

    if features is None:
        continue

    features.update({
        **re_meta,
        "is_bot": 0,
        "bot_type": "human",
    })
    re_rows.append(features)
    re_human_traces.append(re_mouse)

re_games_df = pd.DataFrame(re_rows)
print(re_games_df.head())


## Player identification


In [ ]:
from src.features import feature_cols

select_model = re_games_df

input_data = select_model[feature_cols]
output_data = select_model["userId"]

input_train, input_test, output_train, output_test = train_test_split(
    input_data, output_data, test_size=0.2, random_state=42, stratify=output_data
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(input_train, output_train)

output_pred = model.predict(input_test)
accuracy = accuracy_score(output_test, output_pred)

print(f"Accuracy: {accuracy:.2%}")
print(f"Random baseline: {1 / output_data.nunique():.2%} ({output_data.nunique()} players, {len(select_model)} games)")
print()
print(classification_report(output_test, output_pred))


## Block-bootstrap synthetic bots


In [ ]:
from src.bots import (
    build_segments,
    stitch_bot_game,
    collect_human_motion_samples,
    median_trace_duration_ms,
)
from src.features import extract_features
from src.config import RNG_SEED
from src.data import load_red_eclipse_mouse

rng = np.random.default_rng(RNG_SEED)

# Build segment pool + empirical motion stats from human games
re_segment_pool = []
for file_path in game_files:
    re_meta, re_mouse = load_red_eclipse_mouse(file_path)
    re_segment_pool.extend(build_segments(re_mouse, rng=rng))

# { "dt_samples", "dt_by_session", "step_samples", "angle_samples", "min_step_floor", "step_clip" }
re_motion = collect_human_motion_samples(re_human_traces, rng=rng)
re_target_ms = median_trace_duration_ms(re_human_traces)
print(
    f"Segment pool: {len(re_segment_pool)} segments from {len(game_files)} games | "
    f"dt n={len(re_motion['dt_samples'])} sessions={len(re_motion['dt_by_session'])} median={np.median(re_motion['dt_samples']):.2f}ms | "
    f"step median={np.median(re_motion['step_samples']):.3f} | "
    f"target={re_target_ms/1000:.1f}s"
)

# Generate synthetic bot games
N_PREVIEW = 5 # save first N trajectories for render graph
re_stitch_rows = []
sample_re_stitch_trajectories = []
for i in range(len(re_games_df)):
    bot_mouse = stitch_bot_game(
        re_segment_pool,
        dt_samples=re_motion['dt_samples'],
        dt_by_session=re_motion['dt_by_session'],
        target_duration_ms=re_target_ms,
        rng=rng,
    )
    if len(sample_re_stitch_trajectories) < N_PREVIEW:
        sample_re_stitch_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -1,
        "gameId": f"bot_{i}",
        "source_file": f"synthetic_bot_{i}",
        "is_bot": 1,
        "bot_type": "stitch",
    })
    re_stitch_rows.append(feats)

re_stitch_df = pd.DataFrame(re_stitch_rows)
print(f"Generated {len(re_stitch_df)} stitch bot games")
print(re_stitch_df.head())



## Block-bootstrap synthetic bots trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_re_stitch_trajectories):
    plot_trajectory(df, title=f"bot_{i}")
    print(f"\nbot_{i}, events={len(df)}")


## Smooth bot generation


In [ ]:
from src.bots import (
    estimate_smooth_params,
    generate_smooth_bot_game,
    smooth_generator_params,
    smooth_params_for_print,
)
from src.features import extract_features

re_median_events = int(re_games_df["n_events"].median())

re_smooth_params = estimate_smooth_params(re_games_df, **re_motion)
print(f"RE smooth params: {smooth_params_for_print(re_smooth_params)}")
re_smooth_gen = smooth_generator_params(re_smooth_params)

re_smooth_rows = []
sample_re_smooth_trajectories = []
for i in range(len(re_games_df)):
    bot_mouse = generate_smooth_bot_game(
        n_events=max(re_median_events * 3, 1),
        seed=RNG_SEED + i,
        target_duration_ms=re_target_ms,
        **re_smooth_gen,
    )
    if len(sample_re_smooth_trajectories) < N_PREVIEW:
        sample_re_smooth_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -2,
        "gameId": f"smooth_{i}",
        "source_file": f"synthetic_smooth_{i}",
        "is_bot": 1,
        "bot_type": "smooth",
    })
    re_smooth_rows.append(feats)

re_smooth_df = pd.DataFrame(re_smooth_rows)
print(f"Generated {len(re_smooth_df)} smooth bot games (n_events={re_median_events})")
print(re_smooth_df.head())


## Smooth bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_re_smooth_trajectories):
    plot_trajectory(df, title=f"smooth_{i}")
    print(f"\nsmooth_{i}, events={len(df)}")

## Bézier bot generation


In [ ]:
from src.bots import (
    estimate_bezier_params,
    generate_bezier_bot_game,
    bezier_params_for_print,
)
from src.features import extract_features

N_BEZIER_BOTS = len(re_games_df)

re_bezier_params = estimate_bezier_params(re_games_df, **re_motion)
print(f"RE bezier params: {bezier_params_for_print(re_bezier_params)}")

bezier_rows = []
sample_re_bezier_trajectories = []
for i in range(N_BEZIER_BOTS):
    bot_mouse = generate_bezier_bot_game(
        n_events=max(re_median_events * 3, 1),
        seed=RNG_SEED + 50 + i,
        target_duration_ms=re_target_ms,
        **re_bezier_params,
    )
    if len(sample_re_bezier_trajectories) < N_PREVIEW:
        sample_re_bezier_trajectories.append(bot_mouse.copy())

    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -3,
        "gameId": f"bezier_{i}",
        "source_file": f"synthetic_bezier_{i}",
        "is_bot": 1,
        "bot_type": "bezier",
    })
    bezier_rows.append(feats)

re_bezier_df = pd.DataFrame(bezier_rows)
print(f"Generated {len(re_bezier_df)} bezier bot games (n_events={re_median_events})")
print(re_bezier_df.head())



## Bézier bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_re_bezier_trajectories):
    plot_trajectory(df, title=f"bezier_{i}")
    print(f"\nbezier_{i}, events={len(df)}")



## VAE bot — train once / load weights (RE)


In [ ]:
from pathlib import Path
import numpy as np

from src.config import RNG_SEED
from src.vae_bot import ensure_vae_bundle, DEFAULT_RE_WEIGHTS

# Formal runs: False → load committed weights. True → retrain + overwrite path.
VAE_FORCE_RETRAIN = False
VAE_WEIGHTS_PATH = DEFAULT_RE_WEIGHTS  # artifacts/vae_re_v2.pt (per-axis std)

re_step_median = float(np.median(re_motion["step_samples"]))
re_vae_bundle = ensure_vae_bundle(
    re_human_traces,
    re_step_median,
    path=VAE_WEIGHTS_PATH,
    force_retrain=VAE_FORCE_RETRAIN,
    seed=RNG_SEED,
)
print(
    f"RE VAE ready | path={Path(VAE_WEIGHTS_PATH)} | "
    f"seg_len={re_vae_bundle['seg_len']} z={re_vae_bundle['z_dim']} "
    f"norm={re_vae_bundle.get('norm')} axis_scale={re_vae_bundle.get('axis_scale')} "
    f"trained_segments={re_vae_bundle.get('n_segments')}"
)


## VAE bot generation (RE)


In [ ]:
from src.vae_bot import generate_vae_bot_games
from src.features import extract_features
from src.config import RNG_SEED, VAE_POOL_SEGMENTS

re_vae_rng = np.random.default_rng(RNG_SEED + 3)
re_vae_traces = generate_vae_bot_games(
    re_vae_bundle,
    n_games=len(re_games_df),
    dt_samples=re_motion['dt_samples'],
    dt_by_session=re_motion['dt_by_session'],
    target_duration_ms=re_target_ms,
    n_pool_segments=VAE_POOL_SEGMENTS,
    rng=re_vae_rng,
)
sample_vae_trajectories = [m.copy() for m in re_vae_traces[:N_PREVIEW]]

re_vae_rows = []
for i, bot_mouse in enumerate(re_vae_traces):
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -4,
        "gameId": f"vae_{i}",
        "source_file": f"synthetic_vae_{i}",
        "is_bot": 1,
        "bot_type": "vae",
    })
    re_vae_rows.append(feats)

re_vae_df = pd.DataFrame(re_vae_rows)
print(f"Generated {len(re_vae_df)} VAE bot games (pool={VAE_POOL_SEGMENTS} segs/game)")
print(re_vae_df.head())


## VAE bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_vae_trajectories):
    plot_trajectory(df, title=f"vae_{i}")
    print(f"\nvae_{i}, events={len(df)}")


## RE Human vs Bot classification (GroupKFold + split-player + windows)

In [ ]:
from src.evaluation import evaluate_group_kfold_windows
from src.features import feature_cols
from src.data import load_red_eclipse_mouse
from pathlib import Path

re_file_by_name = {Path(p).name: p for p in game_files}
# read the mouse data from the source file
def _re_traces_for(feat_df):
    return [load_red_eclipse_mouse(re_file_by_name[src])[1] for src in feat_df["source_file"]]

re_cross_result = evaluate_group_kfold_windows(
    human_df=re_games_df,
    groups=re_games_df["userId"].to_numpy(),
    traces_for_df=_re_traces_for,
    feature_cols=feature_cols,
    vae_bundle=re_vae_bundle,
    name="RE",
)
print("\nRE fold table (window counts):")
print(
    re_cross_result["fold_df"][
        [
            "fold",
            "n_train_sessions",
            "n_test_sessions",
            "n_train_windows_human",
            "n_test_windows_human",
            "n_test_groups",
        ]
    ].to_string(index=False)
)
print("\nRE window-level summary:")
print(re_cross_result["summary_df"].to_string(index=False))
if re_cross_result["session_summary_df"] is not None:
    print("\nRE session-mean-of-windows summary:")
    print(re_cross_result["session_summary_df"].to_string(index=False))


## RE train Cross-game transfer features

In [ ]:
from src.features import cross_game_feature_cols, to_scale_invariant, SCALE_INVARIANT_COLS, SCALE_INVARIANT_EXT_COLS
from src.evaluation import train_bot_detector
from src.config import RNG_SEED

print("=== RE model trained on STITCH bots ===")
re_model_stitch, re_acc_stitch = train_bot_detector(re_games_df, re_stitch_df, cross_game_feature_cols, name="stitch")
print()
print("=== RE model trained on SMOOTH bots ===")
re_model_smooth, re_acc_smooth = train_bot_detector(re_games_df, re_smooth_df, cross_game_feature_cols, name="smooth")
print()
print("=== RE model trained on BEZIER bots ===")
re_model_bezier, re_acc_bezier = train_bot_detector(re_games_df, re_bezier_df, cross_game_feature_cols, name="bezier")
print()
print("=== RE model trained on VAE bots ===")
re_model_vae, re_acc_vae = train_bot_detector(re_games_df, re_vae_df, cross_game_feature_cols, name="vae")

# calculate scale invariant features
re_si_human = to_scale_invariant(re_games_df)
re_si_stitch = to_scale_invariant(re_stitch_df)
re_si_smooth = to_scale_invariant(re_smooth_df)
re_si_bezier = to_scale_invariant(re_bezier_df)
re_si_vae = to_scale_invariant(re_vae_df)

# train scale invariant(4) features
print("=== Train on Red Eclipse (scale-invariant features) ===")
m_si_stitch, _ = train_bot_detector(
    re_si_human, re_si_stitch, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant stitch",
    show_feature_importance=False,
)
print()
m_si_smooth, _ = train_bot_detector(
    re_si_human, re_si_smooth, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant smooth",
    show_feature_importance=False,
)
print()
m_si_bezier, _ = train_bot_detector(
    re_si_human, re_si_bezier, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant bezier",
    show_feature_importance=False,
)

print()
m_si_vae, _ = train_bot_detector(
    re_si_human, re_si_vae, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant vae",
    show_feature_importance=False,
)

# train scale invariant(10) features
print("=== Train on Red Eclipse (scale-invariant EXT, 10 feats) ===")
print("cols:", SCALE_INVARIANT_EXT_COLS)
m_si_ext_stitch, _ = train_bot_detector(
    re_si_human, re_si_stitch, SCALE_INVARIANT_EXT_COLS,
    random_state=RNG_SEED, name="SI-EXT stitch",
    show_feature_importance=False,
)
print()
m_si_ext_smooth, _ = train_bot_detector(
    re_si_human, re_si_smooth, SCALE_INVARIANT_EXT_COLS,
    random_state=RNG_SEED, name="SI-EXT smooth",
    show_feature_importance=False,
)
print()
m_si_ext_bezier, _ = train_bot_detector(
    re_si_human, re_si_bezier, SCALE_INVARIANT_EXT_COLS,
    random_state=RNG_SEED, name="SI-EXT bezier",
    show_feature_importance=False,
)
print()
m_si_ext_vae, _ = train_bot_detector(
    re_si_human, re_si_vae, SCALE_INVARIANT_EXT_COLS,
    random_state=RNG_SEED, name="SI-EXT vae",
    show_feature_importance=False,
)

## Load LoL dataset

In [ ]:
from src.data import load_lol_match_windows
from src.config import LOL_MATCH_WINDOW_START_MIN, LOL_MATCH_WINDOW_END_MIN

# first match-aligned window (game clock 10–13 min)
lol_records, lol_load_summary = load_lol_match_windows()
print("LoL match-window summary (smoke):", lol_load_summary)
if lol_records:
    sample_lol = lol_records[0]["mouse"]
    print(f"gameId={lol_records[0]['gameId']}")
    print(
        f"Events: {len(sample_lol)}, duration: {sample_lol['time'].iloc[-1] / 60000:.2f} min "
        f"(target {LOL_MATCH_WINDOW_END_MIN - LOL_MATCH_WINDOW_START_MIN} min @ "
        f"{LOL_MATCH_WINDOW_START_MIN}-{LOL_MATCH_WINDOW_END_MIN})"
    )
    print(sample_lol.head())
else:
    sample_lol = None
    print("No window found for first keylogger (short match or sparse 10–13).")


## LoL extract features

In [ ]:
from src.features import extract_features
from src.config import (
    LOL_MATCH_WINDOW_START_MIN,
    LOL_MATCH_WINDOW_END_MIN,
)

lol_rows = []
lol_human_traces = []
lol_mouse_by_game = {}
for rec in lol_records:
    lol_mouse = rec["mouse"]
    feats = extract_features(lol_mouse)
    if feats is None:
        continue
    lol_meta = {
        "userId": rec["userId"],
        "gameId": rec["gameId"],
        "source_file": rec["source_file"],
        "session_date": rec["session_date"],
        "match_id": rec["match_id"],
        "participant": rec["participant"],
    }
    feats.update({
        **lol_meta,
        "is_bot": 0,
        "bot_type": "human",
    })
    lol_rows.append(feats)
    lol_human_traces.append(lol_mouse)
    lol_mouse_by_game[rec["gameId"]] = lol_mouse

lol_games_df = pd.DataFrame(lol_rows)
print(
    f"Loaded {len(lol_games_df)} LoL human windows "
    f"(match-aligned {LOL_MATCH_WINDOW_START_MIN}–{LOL_MATCH_WINDOW_END_MIN} min)"
)
print(lol_games_df[["n_events", "total_movement", "avg_speed", "idle_ratio"]].describe())
print()
print(f"Red Eclipse — median n_events: {re_games_df['n_events'].median():.0f}")
print(f"LoL — median n_events: {lol_games_df['n_events'].median():.0f}")
print(f"LoL users: {lol_games_df['userId'].nunique()}, matches: {lol_games_df['match_id'].nunique()}")


## LoL trajectory preview

In [ ]:
from src.config import LOL_MATCH_WINDOW_START_MIN, LOL_MATCH_WINDOW_END_MIN
from src.plotting import plot_trajectory

preview_lol = lol_human_traces[0].copy()
print(
    f"Preview: {lol_games_df.iloc[0]['gameId']} | "
    f"{len(preview_lol)} events | {preview_lol['time'].iloc[-1]/60000:.2f} min"
)
plot_trajectory(
    preview_lol,
    title=f"LoL match window {LOL_MATCH_WINDOW_START_MIN}-{LOL_MATCH_WINDOW_END_MIN} min",
)


## LoL stitch bot generation

In [ ]:
from src.bots import (
    build_segments,
    stitch_bot_game,
    collect_human_motion_samples,
    median_trace_duration_ms,
)
from src.features import extract_features

lol_segment_pool = []
if not lol_human_traces:
    raise RuntimeError("lol_human_traces empty — run LoL match-window feature cell first")

lol_bot_rng = np.random.default_rng(RNG_SEED + 1)
for mouse in lol_human_traces:
    lol_segment_pool.extend(build_segments(mouse,
        rng=lol_bot_rng))

# { "dt_samples", "dt_by_session", "step_samples", "angle_samples", "min_step_floor" }
lol_motion = collect_human_motion_samples(lol_human_traces, rng=lol_bot_rng)
lol_target_ms = median_trace_duration_ms(lol_human_traces)
print(
    f"LoL dt n={len(lol_motion['dt_samples'])} sessions={len(lol_motion['dt_by_session'])} median={np.median(lol_motion['dt_samples']):.2f}ms | "
    f"step median={np.median(lol_motion['step_samples']):.3f} | "
    f"target={lol_target_ms/1000:.1f}s"
)

lol_stitch_rows = []
sample_lol_stitch_trajectories = []

for i in range(len(lol_games_df)):
    bot_mouse = stitch_bot_game(
        lol_segment_pool,
        dt_samples=lol_motion['dt_samples'],
        dt_by_session=lol_motion['dt_by_session'],
        target_duration_ms=lol_target_ms,
        rng=lol_bot_rng,
    )
    if len(sample_lol_stitch_trajectories) < N_PREVIEW:
        sample_lol_stitch_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({"userId": -1,
        "gameId": f"lol_stitch_{i}",
        "is_bot": 1,
        "bot_type": "stitch"
    })
    lol_stitch_rows.append(feats)

lol_bots_stitch_df = pd.DataFrame(lol_stitch_rows)
print(f"LoL stitch bots: {len(lol_bots_stitch_df)} (target {lol_target_ms/1000:.1f}s each)")


## LoL stitch bot trajectory preview

In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_lol_stitch_trajectories):
    print(f"lol_stitch_{i}, events={len(df)}")
    plot_trajectory(df, title=f"lol_stitch_{i}")

## LoL smooth bot generation

In [ ]:
from src.bots import (
    estimate_smooth_params,
    generate_smooth_bot_game,
    smooth_generator_params,
    smooth_params_for_print,
)
from src.features import extract_features

lol_median_events = int(lol_games_df["n_events"].median())

lol_smooth_params = estimate_smooth_params(lol_games_df, **lol_motion)
print(f"LoL smooth params: {smooth_params_for_print(lol_smooth_params)}")
print(f"(RE smooth params for comparison: {smooth_params_for_print(re_smooth_params)})")
lol_smooth_gen = smooth_generator_params(lol_smooth_params)

lol_smooth_rows = []
sample_lol_smooth_trajectories = []
for i in range(len(lol_games_df)):
    bot_mouse = generate_smooth_bot_game(
        n_events=max(lol_median_events * 3, 1),
        seed=RNG_SEED + 120 + i,
        target_duration_ms=lol_target_ms,
        **lol_smooth_gen,
    )
    if len(sample_lol_smooth_trajectories) < N_PREVIEW:
        sample_lol_smooth_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({"userId": -2,
        "gameId": f"lol_smooth_{i}",
        "is_bot": 1,
        "bot_type": "smooth"
    })
    lol_smooth_rows.append(feats)

lol_bots_smooth_df = pd.DataFrame(lol_smooth_rows)
print(f"LoL smooth bots: {len(lol_bots_smooth_df)} (n_events={lol_median_events})")


## LoL smooth bot trajectory preview

In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_lol_smooth_trajectories):
    plot_trajectory(df, title=f"lol_smooth_{i}")
    print(f"lol_smooth_{i}, events={len(df)}")


## LoL Bézier bot generation


In [ ]:
from src.bots import (
    estimate_bezier_params,
    generate_bezier_bot_game,
    bezier_params_for_print,
)

lol_median_events = int(lol_games_df["n_events"].median())

lol_bezier_params = estimate_bezier_params(lol_games_df, **lol_motion)
print(f"LoL bezier params: {bezier_params_for_print(lol_bezier_params)}")
print(f"(RE bezier params for comparison: {bezier_params_for_print(re_bezier_params)})")

lol_bezier_rows = []
sample_lol_bezier_trajectories = []
for i in range(len(lol_games_df)):
    bot_mouse = generate_bezier_bot_game(
        n_events=max(lol_median_events * 3, 1),
        seed=RNG_SEED + 150 + i,
        target_duration_ms=lol_target_ms,
        **lol_bezier_params,
    )
    if len(sample_lol_bezier_trajectories) < N_PREVIEW:
        sample_lol_bezier_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({"userId": -3, "gameId": f"lol_bezier_{i}", "is_bot": 1, "bot_type": "bezier"})
    lol_bezier_rows.append(feats)

lol_bots_bezier_df = pd.DataFrame(lol_bezier_rows)
print(f"LoL bezier bots: {len(lol_bots_bezier_df)} (n_events={lol_median_events})")



## LoL Bézier bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_lol_bezier_trajectories):
    plot_trajectory(df, title=f"lol_bezier_{i}")
    print(f"lol_bezier_{i}, events={len(df)}")



## LoL VAE bot — train once / load weights

In [ ]:
from pathlib import Path
import numpy as np

from src.config import RNG_SEED
from src.vae_bot import ensure_vae_bundle, DEFAULT_LOL_WEIGHTS

# False → load artifacts/vae_lol_v4.pt if present; else trains once and saves.
LOL_VAE_FORCE_RETRAIN = False
LOL_VAE_WEIGHTS_PATH = DEFAULT_LOL_WEIGHTS

lol_step_median = float(np.median(lol_motion["step_samples"]))
lol_vae_bundle = ensure_vae_bundle(
    lol_human_traces,
    lol_step_median,
    path=LOL_VAE_WEIGHTS_PATH,
    force_retrain=LOL_VAE_FORCE_RETRAIN,
    seed=RNG_SEED + 1,
)
print(
    f"LoL VAE ready | path={Path(LOL_VAE_WEIGHTS_PATH)} | "
    f"seg_len={lol_vae_bundle['seg_len']} z={lol_vae_bundle['z_dim']} "
    f"norm={lol_vae_bundle.get('norm')} axis_scale={lol_vae_bundle.get('axis_scale')} "
    f"trained_segments={lol_vae_bundle.get('n_segments')}"
)


## LoL VAE bot generation


In [ ]:
from src.vae_bot import generate_vae_bot_games
from src.features import extract_features
from src.config import RNG_SEED, VAE_POOL_SEGMENTS

lol_vae_rng = np.random.default_rng(RNG_SEED + 4)
lol_vae_traces = generate_vae_bot_games(
    lol_vae_bundle,
    n_games=len(lol_games_df),
    dt_samples=lol_motion['dt_samples'],
    dt_by_session=lol_motion['dt_by_session'],
    target_duration_ms=lol_target_ms,
    n_pool_segments=VAE_POOL_SEGMENTS,
    rng=lol_vae_rng,
)
sample_lol_vae_trajectories = [m.copy() for m in lol_vae_traces[:N_PREVIEW]]

lol_vae_rows = []
for i, bot_mouse in enumerate(lol_vae_traces):
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -4,
        "gameId": f"lol_vae_{i}",
        "is_bot": 1,
        "bot_type": "vae",
    })
    lol_vae_rows.append(feats)

lol_bots_vae_df = pd.DataFrame(lol_vae_rows)
print(f"LoL VAE bots: {len(lol_bots_vae_df)} (pool={VAE_POOL_SEGMENTS} segs/game)")
print(lol_bots_vae_df.head())


## LoL VAE bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_lol_vae_trajectories):
    plot_trajectory(df, title=f"lol_vae_{i}")
    print(f"\nlol_vae_{i}, events={len(df)}")


## LoL in-domain (GroupKFold + split-player + windows)

In [ ]:
from src.evaluation import evaluate_group_kfold_windows
from src.features import feature_cols
from src.config import RNG_SEED

def _lol_traces_for(feat_df):
    return [lol_mouse_by_game[gid] for gid in feat_df["gameId"]]

lol_cross_result = evaluate_group_kfold_windows(
    human_df=lol_games_df,
    groups=lol_games_df["userId"].to_numpy(),
    traces_for_df=_lol_traces_for,
    feature_cols=feature_cols,
    rng_seed_base=RNG_SEED + 1,
    vae_bundle=lol_vae_bundle,
    name="LoL",
)
print("\nLoL fold table (window counts):")
print(
    lol_cross_result["fold_df"][
        [
            "fold",
            "n_train_sessions",
            "n_test_sessions",
            "n_train_windows_human",
            "n_test_windows_human",
            "n_test_groups",
        ]
    ].to_string(index=False)
)
print("\nLoL window-level summary:")
print(lol_cross_result["summary_df"].to_string(index=False))
if lol_cross_result["session_summary_df"] is not None:
    print("\nLoL session-mean-of-windows summary:")
    print(lol_cross_result["session_summary_df"].to_string(index=False))


## Feature scale comparison (RE vs LoL)

In [ ]:
from src.features import cross_game_feature_cols

cols = cross_game_feature_cols

print("Feature medians (RE human vs LoL human vs LoL bots):")
compare = pd.DataFrame({
    "RE_human": re_games_df[cols].median(),
    "LoL_human": lol_games_df[cols].median(),
    "LoL_stitch": lol_bots_stitch_df[cols].median(),
    "LoL_smooth": lol_bots_smooth_df[cols].median(),
    "LoL_bezier": lol_bots_bezier_df[cols].median(),
    "LoL_vae": lol_bots_vae_df[cols].median(),
}).round(3)
print(compare)


## RE → LoL diagnose (raw features)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import cross_game_feature_cols

print("=== Raw features: true zero-shot RE -> LoL ===")
print("(threshold from LoL humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", re_model_stitch, lol_games_df, lol_bots_stitch_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "smooth", re_model_smooth, lol_games_df, lol_bots_smooth_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "bezier", re_model_bezier, lol_games_df, lol_bots_bezier_df,
    cross_game_feature_cols, title_suffix="raw",
)

print()
_ = diagnose_cross_game(
    "vae", re_model_vae, lol_games_df, lol_bots_vae_df,
    cross_game_feature_cols, title_suffix="raw",
)


## RE → LoL diagnose (scale-invariant features)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS

lol_si_human = to_scale_invariant(lol_games_df)
lol_si_stitch = to_scale_invariant(lol_bots_stitch_df)
lol_si_smooth = to_scale_invariant(lol_bots_smooth_df)
lol_si_bezier = to_scale_invariant(lol_bots_bezier_df)
lol_si_vae = to_scale_invariant(lol_bots_vae_df)

print("=== Scale-invariant: true zero-shot RE -> LoL ===")
print("(threshold from LoL humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_si_stitch, lol_si_human, lol_si_stitch,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "smooth", m_si_smooth, lol_si_human, lol_si_smooth,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "bezier", m_si_bezier, lol_si_human, lol_si_bezier,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)

print()
_ = diagnose_cross_game(
    "vae", m_si_vae, lol_si_human, lol_si_vae,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)


## RE → LoL diagnose (scale-invariant EXT)


In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import SCALE_INVARIANT_EXT_COLS

print("=== Scale-invariant EXT (10 feats): true zero-shot RE -> LoL ===")
print("(threshold from LoL humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_si_ext_stitch, lol_si_human, lol_si_stitch,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)
print()
_ = diagnose_cross_game(
    "smooth", m_si_ext_smooth, lol_si_human, lol_si_smooth,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)
print()
_ = diagnose_cross_game(
    "bezier", m_si_ext_bezier, lol_si_human, lol_si_bezier,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)
print()
_ = diagnose_cross_game(
    "vae", m_si_ext_vae, lol_si_human, lol_si_vae,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)


## SI-EXT ablation: drop `xy_corr` / `vh_ratio` (RE → LoL)

In [ ]:
import pandas as pd
from src.evaluation import train_bot_detector, diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS
from src.config import RNG_SEED

EXT_FULL = list(SCALE_INVARIANT_EXT_COLS)
EXT_NO_XY = [c for c in EXT_FULL if c != "xy_corr"]
EXT_NO_XY_VH = [c for c in EXT_NO_XY if c != "vh_ratio"]

re_si_human = to_scale_invariant(re_games_df)
re_bots = {
    "stitch": to_scale_invariant(re_stitch_df),
    "smooth": to_scale_invariant(re_smooth_df),
    "bezier": to_scale_invariant(re_bezier_df),
    "vae": to_scale_invariant(re_vae_df),
}
lol_si_human = to_scale_invariant(lol_games_df)
lol_bots = {
    "stitch": to_scale_invariant(lol_bots_stitch_df),
    "smooth": to_scale_invariant(lol_bots_smooth_df),
    "bezier": to_scale_invariant(lol_bots_bezier_df),
    "vae": to_scale_invariant(lol_bots_vae_df),
}

ablations = {
    "EXT-10": EXT_FULL,
    "EXT-xy": EXT_NO_XY,
    "EXT-xy-vh": EXT_NO_XY_VH,
}

rows = []
for abl_name, cols in ablations.items():
    print(f"\n{'=' * 60}")
    print(f"=== Ablation {abl_name} ({len(cols)} feats): {cols} ===")
    for bot in ["stitch", "smooth", "bezier", "vae"]:
        model, _ = train_bot_detector(
            re_si_human, re_bots[bot], cols,
            random_state=RNG_SEED, name=f"{abl_name} {bot}",
            show_feature_importance=False,
        )
        d = diagnose_cross_game(
            bot, model, lol_si_human, lol_bots[bot], cols,
            title_suffix=f"RE→LoL {abl_name}",
            plot=False, plot_proba=False,
        )
        rows.append({
            "ablation": abl_name,
            "n_feats": len(cols),
            "bot": bot,
            "auc": d["auc"],
            "cal_detect": d["detect_cal"],
            "cal_fp": d["fp_cal"],
            "thr": d["thr"],
            "detect_05": d["detect_05"],
            "fp_05": d["fp_05"],
        })
        print()

summary = pd.DataFrame(rows)
print("\n=== RE→LoL SI-EXT ablation summary ===")
print(
    summary.pivot_table(
        index="bot", columns="ablation",
        values=["auc", "cal_detect"],
    ).round(3).to_string()
)
print("\n(VAE focus: does cal_detect stay high after dropping vh_ratio?)")
print(summary[summary["bot"] == "vae"][
    ["ablation", "n_feats", "auc", "cal_detect", "cal_fp", "thr"]
].to_string(index=False))


## RE→LoL Bézier inversion diagnostic


In [ ]:
import pandas as pd
from src.features import cross_game_feature_cols, to_scale_invariant, SCALE_INVARIANT_COLS

raw_cols = cross_game_feature_cols

# --- (1) Feature medians: RE human / RE bezier / LoL human / LoL bezier ---
print("=== Raw 7-feature medians ===")
raw_med = pd.DataFrame({
    "RE_human": re_games_df[raw_cols].median(),
    "RE_bezier": re_bezier_df[raw_cols].median(),
    "LoL_human": lol_games_df[raw_cols].median(),
    "LoL_bezier": lol_bots_bezier_df[raw_cols].median(),
}).round(4)
# signed gap human - bot (same game); flip if RE and LoL gaps have opposite sign
raw_med["gap_RE"] = (raw_med["RE_human"] - raw_med["RE_bezier"]).round(4)
raw_med["gap_LoL"] = (raw_med["LoL_human"] - raw_med["LoL_bezier"]).round(4)
raw_med["sign_flip"] = (raw_med["gap_RE"] * raw_med["gap_LoL"]) < 0
print(raw_med)
print()

re_sf = to_scale_invariant(re_games_df)
re_bz_sf = to_scale_invariant(re_bezier_df)
lol_sf = to_scale_invariant(lol_games_df)
lol_bz_sf = to_scale_invariant(lol_bots_bezier_df)

print("=== Scale-invariant 4-feature medians ===")
sf_med = pd.DataFrame({
    "RE_human": re_sf[SCALE_INVARIANT_COLS].median(),
    "RE_bezier": re_bz_sf[SCALE_INVARIANT_COLS].median(),
    "LoL_human": lol_sf[SCALE_INVARIANT_COLS].median(),
    "LoL_bezier": lol_bz_sf[SCALE_INVARIANT_COLS].median(),
}).round(4)
sf_med["gap_RE"] = (sf_med["RE_human"] - sf_med["RE_bezier"]).round(4)
sf_med["gap_LoL"] = (sf_med["LoL_human"] - sf_med["LoL_bezier"]).round(4)
sf_med["sign_flip"] = (sf_med["gap_RE"] * sf_med["gap_LoL"]) < 0
print(sf_med)
print()

# --- (2) Feature importance of the RE-trained bezier detectors ---
print("=== RE bezier model importance (raw 7-feat, used in RE→LoL raw) ===")
imp_raw = pd.Series(
    re_model_bezier.feature_importances_, index=raw_cols
).sort_values(ascending=False)
print(imp_raw.round(4))
print()

print("=== RE bezier model importance (scale-invariant, used in RE→LoL SF) ===")
imp_sf = pd.Series(
    m_si_bezier.feature_importances_, index=SCALE_INVARIANT_COLS
).sort_values(ascending=False)
print(imp_sf.round(4))
print()

# --- Cross-read: high importance ∩ sign flip ---
print("=== Suspects: importance rank + sign_flip ===")
print("Raw:")
for feat, imp in imp_raw.items():
    flip = bool(raw_med.loc[feat, "sign_flip"])
    print(f"  {feat:16s}  imp={imp:.3f}  flip={flip}  "
          f"gap_RE={raw_med.loc[feat, 'gap_RE']:+.4f}  gap_LoL={raw_med.loc[feat, 'gap_LoL']:+.4f}")
print("Scale-invariant:")
for feat, imp in imp_sf.items():
    flip = bool(sf_med.loc[feat, "sign_flip"])
    print(f"  {feat:16s}  imp={imp:.3f}  flip={flip}  "
          f"gap_RE={sf_med.loc[feat, 'gap_RE']:+.4f}  gap_LoL={sf_med.loc[feat, 'gap_LoL']:+.4f}")




## RE→LoL Bézier diagnostic (scale-invariant EXT medians)


In [ ]:
import pandas as pd
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS

re_sf = to_scale_invariant(re_games_df)
re_bz_sf = to_scale_invariant(re_bezier_df)
lol_sf = to_scale_invariant(lol_games_df)
lol_bz_sf = to_scale_invariant(lol_bots_bezier_df)

print("=== Scale-invariant EXT 10-feature medians ===")
sf_med = pd.DataFrame({
    "RE_human": re_sf[SCALE_INVARIANT_EXT_COLS].median(),
    "RE_bezier": re_bz_sf[SCALE_INVARIANT_EXT_COLS].median(),
    "LoL_human": lol_sf[SCALE_INVARIANT_EXT_COLS].median(),
    "LoL_bezier": lol_bz_sf[SCALE_INVARIANT_EXT_COLS].median(),
}).round(4)
sf_med["gap_RE"] = (sf_med["RE_human"] - sf_med["RE_bezier"]).round(4)
sf_med["gap_LoL"] = (sf_med["LoL_human"] - sf_med["LoL_bezier"]).round(4)
sf_med["sign_flip"] = (sf_med["gap_RE"] * sf_med["gap_LoL"]) < 0
print(sf_med)
print()

print("=== RE bezier model importance (SI-EXT, used in RE→LoL EXT) ===")
imp_sf = pd.Series(
    m_si_ext_bezier.feature_importances_, index=SCALE_INVARIANT_EXT_COLS
).sort_values(ascending=False)
print(imp_sf.round(4))
print()

print("=== Suspects: SI-EXT importance + sign_flip ===")
for feat, imp in imp_sf.items():
    flip = bool(sf_med.loc[feat, "sign_flip"])
    print(f"  {feat:16s}  imp={imp:.3f}  flip={flip}  "
          f"gap_RE={sf_med.loc[feat, 'gap_RE']:+.4f}  gap_LoL={sf_med.loc[feat, 'gap_LoL']:+.4f}")


## Feature importance (scale-invariant models)



In [ ]:
import pandas as pd
from src.features import SCALE_INVARIANT_COLS, SCALE_INVARIANT_EXT_COLS

imp = pd.Series(m_si_stitch.feature_importances_, index=SCALE_INVARIANT_COLS).sort_values(ascending=False)
print("=== Feature importance (scale-invariant stitch model) ===")
print(imp)
print()
imp2 = pd.Series(m_si_smooth.feature_importances_, index=SCALE_INVARIANT_COLS).sort_values(ascending=False)
print("=== Feature importance (scale-invariant smooth model) ===")
print(imp2)
print()
imp3 = pd.Series(m_si_bezier.feature_importances_, index=SCALE_INVARIANT_COLS).sort_values(ascending=False)
print("=== Feature importance (scale-invariant bezier model) ===")
print(imp3)
print()
imp4 = pd.Series(m_si_vae.feature_importances_, index=SCALE_INVARIANT_COLS).sort_values(ascending=False)
print("=== Feature importance (scale-invariant vae model) ===")
print(imp4)

imp_ext = pd.Series(m_si_ext_stitch.feature_importances_, index=SCALE_INVARIANT_EXT_COLS).sort_values(ascending=False)
print("=== Feature importance (SI-EXT stitch) ===")
print(imp_ext)
print()
imp_ext2 = pd.Series(m_si_ext_smooth.feature_importances_, index=SCALE_INVARIANT_EXT_COLS).sort_values(ascending=False)
print("=== Feature importance (SI-EXT smooth) ===")
print(imp_ext2)
print()
imp_ext3 = pd.Series(m_si_ext_bezier.feature_importances_, index=SCALE_INVARIANT_EXT_COLS).sort_values(ascending=False)
print("=== Feature importance (SI-EXT bezier) ===")
print(imp_ext3)
print()
imp_ext4 = pd.Series(m_si_ext_vae.feature_importances_, index=SCALE_INVARIANT_EXT_COLS).sort_values(ascending=False)
print("=== Feature importance (SI-EXT vae) ===")
print(imp_ext4)

## CSGO load: eye_vector → (dx, dy, time)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from src.config import CSGO_DATA_ROOT
from src.data_csgo import (
    check_axis_convention,
    eye_vectors_to_mouse_df,
    validate_real_roundtrip,
)

USE_COLS = ["time", "eyeVectorX", "eyeVectorY", "eyeVectorZ"]

def find_gameflt_files(root=CSGO_DATA_ROOT):
    files = sorted(Path(root).rglob("gameFlt.csv"))
    print(f"Found {len(files)} gameFlt.csv under {root}")
    return files

def load_eye_csv(path):
    df = pd.read_csv(path, usecols=USE_COLS)
    finite = np.isfinite(df[["eyeVectorX", "eyeVectorY", "eyeVectorZ"]]).all(axis=1)
    return df.loc[finite].reset_index(drop=True)

def gameflt_to_mouse(path):
    flt = load_eye_csv(path)
    mouse_df, meta = eye_vectors_to_mouse_df(
        flt["time"], flt["eyeVectorX"], flt["eyeVectorY"], flt["eyeVectorZ"]
    )
    return mouse_df, meta, flt

gameflt_paths = find_gameflt_files()
assert gameflt_paths, f"No gameFlt.csv under {CSGO_DATA_ROOT}"

first_path = gameflt_paths[0]
print(f"\n=== First-file checks: {first_path} ===")

mouse0, meta0, flt0 = gameflt_to_mouse(first_path)
print("convert meta:", meta0)
print(mouse0.head())
# check Y=pitch and angles round-trip
axis_ok = check_axis_convention(flt0["eyeVectorY"])
roundtrip = validate_real_roundtrip(
    flt0["eyeVectorX"], flt0["eyeVectorY"], flt0["eyeVectorZ"]
)

if not (axis_ok and roundtrip["ok"]):
    raise RuntimeError(
        "First-file checks failed — stop before converting all files. "
            f"axis_ok={axis_ok}, roundtrip_ok={roundtrip['ok']}"
        )

print("\nFirst file PASSED. Converting all gameFlt.csv ...")

csgo_mouse = {}
rows = []
for i, path in enumerate(gameflt_paths, 1):
    # path like .../S001/P3/gameFlt.csv
    participant = path.parent.name          # P3
    session = path.parent.parent.name       # S001
    key = (session, participant)
    try:
        mouse_df, meta, _ = gameflt_to_mouse(path)
    except Exception as e:
        print(f"  SKIP {key}: {e}")
        continue
    csgo_mouse[key] = mouse_df
    rows.append({
        "session": session,
        "participant": participant,
        "n_out": meta["n_out"],
        "teleport_frac": meta["teleport_frac"],
        "path": str(path),
    })
    if i % 50 == 0 or i == len(gameflt_paths):
        print(f"  converted {i}/{len(gameflt_paths)}")

csgo_convert_summary = pd.DataFrame(rows)
print(f"\nDone: {len(csgo_mouse)} traces")
print(csgo_convert_summary.head())
print(
    "n_out median:", csgo_convert_summary["n_out"].median(),
    "| teleport_frac median:", f"{csgo_convert_summary['teleport_frac'].median():.2%}",
)

## CSGO sessions & extract features


In [ ]:
from pathlib import Path

from src.features import extract_features
from src.config import CSGO_DATA_ROOT, CSGO_WINDOW_MIN
from src.data_csgo import window_mouse_round_alive

print(
    f"Extracting features from {len(csgo_mouse)} CSGO traces "
    f"(Round2+alive, {CSGO_WINDOW_MIN} min)"
)

csgo_mouse_win = {}
csgo_rows = []
csgo_window_meta = []

for (session, participant), mouse in csgo_mouse.items():
    session_dir = Path(CSGO_DATA_ROOT) / session / participant
    csgo_win, meta = window_mouse_round_alive(
        mouse, session_dir, window_min=CSGO_WINDOW_MIN
    )
    csgo_window_meta.append({"session": session, "participant": participant, **meta})

    feats = extract_features(csgo_win)

    csgo_mouse_win[(session, participant)] = csgo_win
    feats.update({
        "userId": f"csgo_{session}_{participant}",
        "gameId": f"{session}_{participant}",
        "session": session,
        "participant": participant,
        "is_bot": 0,
        "bot_type": "human",
        "round_n": meta["round_n"],
        "alive_frac_in_window": meta["alive_frac_in_window"],
    })
    csgo_rows.append(feats)

csgo_games_df = pd.DataFrame(csgo_rows)
csgo_window_meta_df = pd.DataFrame(csgo_window_meta)

print(f"Loaded {len(csgo_games_df)} CSGO human sessions ")
print(
    "alive_frac_in_window median:",
    f"{csgo_games_df['alive_frac_in_window'].median():.1%}",
)
print(csgo_games_df[["n_events", "total_movement", "avg_speed", "idle_ratio"]].describe())
print()
print(f"Red Eclipse — median n_events: {re_games_df['n_events'].median():.0f}")
print(f"LoL — median n_events: {lol_games_df['n_events'].median():.0f}")
print(f"CSGO — median n_events: {csgo_games_df['n_events'].median():.0f}")
print(csgo_games_df.head())


## CSGO trajectory preview


In [ ]:
from src.plotting import plot_trajectory
from src.config import CSGO_WINDOW_MIN

preview_keys = list(csgo_mouse_win.keys())[:5]

for session, participant in preview_keys:
    mouse = csgo_mouse_win[(session, participant)]
    plot_trajectory(
        mouse,
        title=f"CSGO {session}/{participant} (Round2+alive, {CSGO_WINDOW_MIN} min)",
    )
    print(f"\n{session}/{participant}, events={len(mouse)}")


## CSGO stitch bot generation


In [ ]:
from src.bots import (
    build_segments,
    stitch_bot_game,
    collect_human_motion_samples,
    median_trace_duration_ms,
)
from src.features import extract_features
from src.config import RNG_SEED

csgo_bot_rng = np.random.default_rng(RNG_SEED + 2)
csgo_segment_pool = []
for mouse in csgo_mouse_win.values():
    csgo_segment_pool.extend(build_segments(mouse, rng=csgo_bot_rng))

# { "dt_samples", "dt_by_session", "step_samples", "angle_samples", "min_step_floor", "step_clip" }
csgo_motion = collect_human_motion_samples(csgo_mouse_win.values(), rng=csgo_bot_rng)
csgo_target_ms = median_trace_duration_ms(csgo_mouse_win.values())
print(
    f"CSGO segment pool: {len(csgo_segment_pool)} segments "
    f"from {len(csgo_mouse_win)} Round2+alive traces | "
    f"dt n={len(csgo_motion['dt_samples'])} sessions={len(csgo_motion['dt_by_session'])} median={np.median(csgo_motion['dt_samples']):.2f}ms | "
    f"step median={np.median(csgo_motion['step_samples']):.3f} | "
    f"target={csgo_target_ms/1000:.1f}s"
)

N_PREVIEW = 5
csgo_stitch_rows = []
sample_csgo_stitch_trajectories = []

for i in range(len(csgo_games_df)):
    bot_mouse = stitch_bot_game(
        csgo_segment_pool,
        dt_samples=csgo_motion['dt_samples'],
        dt_by_session=csgo_motion['dt_by_session'],
        target_duration_ms=csgo_target_ms,
        rng=csgo_bot_rng,
    )
    if len(sample_csgo_stitch_trajectories) < N_PREVIEW:
        sample_csgo_stitch_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -1,
        "gameId": f"csgo_stitch_{i}",
        "is_bot": 1,
        "bot_type": "stitch",
    })
    csgo_stitch_rows.append(feats)

csgo_bots_stitch_df = pd.DataFrame(csgo_stitch_rows)
print(f"CSGO stitch bots: {len(csgo_bots_stitch_df)} (target {csgo_target_ms/1000:.1f}s each)")
print(csgo_bots_stitch_df.head())



## CSGO stitch bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_csgo_stitch_trajectories):
    print(f"csgo_stitch_{i}, events={len(df)}")
    plot_trajectory(df, title=f"csgo_stitch_{i}")


## CSGO smooth bot generation


In [ ]:
from src.bots import (
    estimate_smooth_params,
    generate_smooth_bot_game,
    smooth_generator_params,
    smooth_params_for_print,
)
from src.features import extract_features
from src.config import RNG_SEED

csgo_median_events = int(csgo_games_df["n_events"].median())

csgo_smooth_params = estimate_smooth_params(csgo_games_df, **csgo_motion)
print(f"CSGO smooth params: {smooth_params_for_print(csgo_smooth_params)}")
csgo_smooth_gen = smooth_generator_params(csgo_smooth_params)

csgo_smooth_rows = []
sample_csgo_smooth_trajectories = []
for i in range(len(csgo_games_df)):
    # round_deltas=False: CSGO dx/dy are degrees (often << 1); integer round would wipe them
    bot_mouse = generate_smooth_bot_game(
        n_events=max(csgo_median_events * 3, 1),
        seed=RNG_SEED + 200 + i,
        round_deltas=False,
        target_duration_ms=csgo_target_ms,
        **csgo_smooth_gen,
    )
    if len(sample_csgo_smooth_trajectories) < N_PREVIEW:
        sample_csgo_smooth_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -2,
        "gameId": f"csgo_smooth_{i}",
        "is_bot": 1,
        "bot_type": "smooth",
    })
    csgo_smooth_rows.append(feats)

csgo_bots_smooth_df = pd.DataFrame(csgo_smooth_rows)
print(f"CSGO smooth bots: {len(csgo_bots_smooth_df)} (n_events={csgo_median_events})")
print(csgo_bots_smooth_df.head())


## CSGO smooth bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_csgo_smooth_trajectories):
    plot_trajectory(df, title=f"csgo_smooth_{i}")
    print(f"csgo_smooth_{i}, events={len(df)}")


## CSGO Bézier bot generation


In [ ]:
from src.bots import (
    estimate_bezier_params,
    generate_bezier_bot_game,
    bezier_params_for_print,
)
from src.features import extract_features
from src.config import RNG_SEED

csgo_median_events = int(csgo_games_df["n_events"].median())

csgo_bezier_params = estimate_bezier_params(csgo_games_df, **csgo_motion)
print(f"CSGO bezier params: {bezier_params_for_print(csgo_bezier_params)}")

csgo_bezier_rows = []
sample_csgo_bezier_trajectories = []
for i in range(len(csgo_games_df)):
    # round_deltas=False: CSGO dx/dy are degrees (often << 1); integer round would wipe them
    bot_mouse = generate_bezier_bot_game(
        n_events=max(csgo_median_events * 3, 1),
        seed=RNG_SEED + 250 + i,
        round_deltas=False,
        target_duration_ms=csgo_target_ms,
        **csgo_bezier_params,
    )
    if len(sample_csgo_bezier_trajectories) < N_PREVIEW:
        sample_csgo_bezier_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -3,
        "gameId": f"csgo_bezier_{i}",
        "is_bot": 1,
        "bot_type": "bezier",
    })
    csgo_bezier_rows.append(feats)

csgo_bots_bezier_df = pd.DataFrame(csgo_bezier_rows)
print(f"CSGO bezier bots: {len(csgo_bots_bezier_df)} (n_events={csgo_median_events})")
print(csgo_bots_bezier_df.head())



## CSGO Bézier bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_csgo_bezier_trajectories):
    plot_trajectory(df, title=f"csgo_bezier_{i}")
    print(f"csgo_bezier_{i}, events={len(df)}")



## CSGO VAE bot — train once / load weights


In [ ]:
from pathlib import Path
import numpy as np

from src.config import RNG_SEED
from src.vae_bot import ensure_vae_bundle, DEFAULT_CSGO_WEIGHTS

CSGO_VAE_FORCE_RETRAIN = False
CSGO_VAE_WEIGHTS_PATH = DEFAULT_CSGO_WEIGHTS

csgo_step_median = float(np.median(csgo_motion["step_samples"]))
csgo_vae_bundle = ensure_vae_bundle(
    list(csgo_mouse_win.values()),
    csgo_step_median,
    path=CSGO_VAE_WEIGHTS_PATH,
    force_retrain=CSGO_VAE_FORCE_RETRAIN,
    seed=RNG_SEED + 2,
)
print(
    f"CSGO VAE ready | path={Path(CSGO_VAE_WEIGHTS_PATH)} | "
    f"seg_len={csgo_vae_bundle['seg_len']} z={csgo_vae_bundle['z_dim']} "
    f"norm={csgo_vae_bundle.get('norm')} axis_scale={csgo_vae_bundle.get('axis_scale')} "
    f"trained_segments={csgo_vae_bundle.get('n_segments')}"
)


## CSGO VAE bot generation


In [ ]:
from src.vae_bot import generate_vae_bot_games
from src.features import extract_features
from src.config import RNG_SEED, VAE_POOL_SEGMENTS

csgo_vae_rng = np.random.default_rng(RNG_SEED + 5)
csgo_vae_traces = generate_vae_bot_games(
    csgo_vae_bundle,
    n_games=len(csgo_games_df),
    dt_samples=csgo_motion['dt_samples'],
    dt_by_session=csgo_motion['dt_by_session'],
    target_duration_ms=csgo_target_ms,
    n_pool_segments=VAE_POOL_SEGMENTS,
    rng=csgo_vae_rng,
)
sample_csgo_vae_trajectories = [m.copy() for m in csgo_vae_traces[:N_PREVIEW]]

csgo_vae_rows = []
for i, bot_mouse in enumerate(csgo_vae_traces):
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -4,
        "gameId": f"csgo_vae_{i}",
        "is_bot": 1,
        "bot_type": "vae",
    })
    csgo_vae_rows.append(feats)

csgo_bots_vae_df = pd.DataFrame(csgo_vae_rows)
print(f"CSGO VAE bots: {len(csgo_bots_vae_df)} (pool={VAE_POOL_SEGMENTS} segs/game)")
print(csgo_bots_vae_df.head())


## CSGO VAE bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_csgo_vae_trajectories):
    plot_trajectory(df, title=f"csgo_vae_{i}")
    print(f"\ncsgo_vae_{i}, events={len(df)}")


## CSGO in-domain (GroupKFold + split-player + **windows**)

In [ ]:
from src.evaluation import evaluate_group_kfold_windows
from src.features import feature_cols
from src.config import RNG_SEED

def _csgo_traces_for(feat_df):
    return [
        csgo_mouse_win[(row.session, row.participant)]
        for row in feat_df.itertuples(index=False)
    ]

csgo_cv = evaluate_group_kfold_windows(
    human_df=csgo_games_df,
    groups=csgo_games_df["participant"].to_numpy(),
    traces_for_df=_csgo_traces_for,
    feature_cols=feature_cols,
    rng_seed_base=RNG_SEED + 2,
    round_deltas=False,
    vae_bundle=csgo_vae_bundle,
    name="CSGO",
)
csgo_cv_summary = csgo_cv["summary_df"]
csgo_cv_session_summary = csgo_cv["session_summary_df"]
csgo_cv_folds = csgo_cv["fold_df"]
print("\nCSGO fold table (window counts):")
print(
    csgo_cv_folds[
        [
            "fold",
            "n_train_sessions",
            "n_test_sessions",
            "n_train_windows_human",
            "n_test_windows_human",
            "n_test_groups",
        ]
    ].to_string(index=False)
)
print("\nCSGO window-level summary:")
print(csgo_cv_summary.to_string(index=False))
if csgo_cv_session_summary is not None:
    print("\nCSGO session-mean-of-windows summary:")
    print(csgo_cv_session_summary.to_string(index=False))


## Zero-shot diagnose (raw features, RE → CSGO)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import cross_game_feature_cols

print("=== Raw features: true zero-shot RE -> CSGO ===")
print("(threshold from CSGO humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", re_model_stitch, csgo_games_df, csgo_bots_stitch_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "smooth", re_model_smooth, csgo_games_df, csgo_bots_smooth_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "bezier", re_model_bezier, csgo_games_df, csgo_bots_bezier_df,
    cross_game_feature_cols, title_suffix="raw",
)

print()
_ = diagnose_cross_game(
    "vae", re_model_vae, csgo_games_df, csgo_bots_vae_df,
    cross_game_feature_cols, title_suffix="raw",
)


## Scale-invariant train (RE) — for CSGO transfer


In [ ]:
from src.evaluation import train_bot_detector
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS
from src.config import RNG_SEED

re_si_human = to_scale_invariant(re_games_df)
re_si_stitch = to_scale_invariant(re_stitch_df)
re_si_smooth = to_scale_invariant(re_smooth_df)
re_si_bezier = to_scale_invariant(re_bezier_df)
re_si_vae = to_scale_invariant(re_vae_df)

print("=== Train on Red Eclipse (scale-invariant features) ===")
m_si_stitch, _ = train_bot_detector(
    re_si_human, re_si_stitch, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant stitch",
    show_feature_importance=False,
)
print()
m_si_smooth, _ = train_bot_detector(
    re_si_human, re_si_smooth, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant smooth",
    show_feature_importance=False,
)
print()
m_si_bezier, _ = train_bot_detector(
    re_si_human, re_si_bezier, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant bezier",
    show_feature_importance=False,
)

print()
m_si_vae, _ = train_bot_detector(
    re_si_human, re_si_vae, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant vae",
    show_feature_importance=False,
)


## Scale-invariant EXT train (RE, 10 feats) — for CSGO transfer


In [ ]:
from src.evaluation import train_bot_detector
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS
from src.config import RNG_SEED

re_si_human = to_scale_invariant(re_games_df)
re_si_stitch = to_scale_invariant(re_stitch_df)
re_si_smooth = to_scale_invariant(re_smooth_df)
re_si_bezier = to_scale_invariant(re_bezier_df)
re_si_vae = to_scale_invariant(re_vae_df)

print("=== Train on Red Eclipse (scale-invariant EXT, 10 feats) — for CSGO transfer ===")
print("cols:", SCALE_INVARIANT_EXT_COLS)
m_si_ext_stitch, _ = train_bot_detector(
    re_si_human, re_si_stitch, SCALE_INVARIANT_EXT_COLS,
    random_state=RNG_SEED, name="SI-EXT stitch",
    show_feature_importance=False,
)
print()
m_si_ext_smooth, _ = train_bot_detector(
    re_si_human, re_si_smooth, SCALE_INVARIANT_EXT_COLS,
    random_state=RNG_SEED, name="SI-EXT smooth",
    show_feature_importance=False,
)
print()
m_si_ext_bezier, _ = train_bot_detector(
    re_si_human, re_si_bezier, SCALE_INVARIANT_EXT_COLS,
    random_state=RNG_SEED, name="SI-EXT bezier",
    show_feature_importance=False,
)
print()
m_si_ext_vae, _ = train_bot_detector(
    re_si_human, re_si_vae, SCALE_INVARIANT_EXT_COLS,
    random_state=RNG_SEED, name="SI-EXT vae",
    show_feature_importance=False,
)


## Zero-shot diagnose (scale-invariant features, RE → CSGO)


In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS

csgo_si_human = to_scale_invariant(csgo_games_df)
csgo_si_stitch = to_scale_invariant(csgo_bots_stitch_df)
csgo_si_smooth = to_scale_invariant(csgo_bots_smooth_df)
csgo_si_bezier = to_scale_invariant(csgo_bots_bezier_df)
csgo_si_vae = to_scale_invariant(csgo_bots_vae_df)

print("=== Scale-invariant: true zero-shot RE -> CSGO ===")
print("(threshold from CSGO humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_si_stitch, csgo_si_human, csgo_si_stitch,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "smooth", m_si_smooth, csgo_si_human, csgo_si_smooth,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "bezier", m_si_bezier, csgo_si_human, csgo_si_bezier,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)

print()
_ = diagnose_cross_game(
    "vae", m_si_vae, csgo_si_human, csgo_si_vae,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)


## Zero-shot diagnose (scale-invariant EXT, RE → CSGO)


In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS

csgo_si_human = to_scale_invariant(csgo_games_df)
csgo_si_stitch = to_scale_invariant(csgo_bots_stitch_df)
csgo_si_smooth = to_scale_invariant(csgo_bots_smooth_df)
csgo_si_bezier = to_scale_invariant(csgo_bots_bezier_df)
csgo_si_vae = to_scale_invariant(csgo_bots_vae_df)

print("=== Scale-invariant EXT (10 feats): true zero-shot RE -> CSGO ===")
print("(threshold from CSGO humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_si_ext_stitch, csgo_si_human, csgo_si_stitch,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)
print()
_ = diagnose_cross_game(
    "smooth", m_si_ext_smooth, csgo_si_human, csgo_si_smooth,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)
print()
_ = diagnose_cross_game(
    "bezier", m_si_ext_bezier, csgo_si_human, csgo_si_bezier,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)
print()
_ = diagnose_cross_game(
    "vae", m_si_ext_vae, csgo_si_human, csgo_si_vae,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)


## SI-EXT ablation: drop `xy_corr` / `vh_ratio` (RE → CSGO)


In [ ]:
import pandas as pd
from src.evaluation import train_bot_detector, diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS
from src.config import RNG_SEED

EXT_FULL = list(SCALE_INVARIANT_EXT_COLS)
EXT_NO_XY = [c for c in EXT_FULL if c != "xy_corr"]
EXT_NO_XY_VH = [c for c in EXT_NO_XY if c != "vh_ratio"]

re_si_human = to_scale_invariant(re_games_df)
re_bots = {
    "stitch": to_scale_invariant(re_stitch_df),
    "smooth": to_scale_invariant(re_smooth_df),
    "bezier": to_scale_invariant(re_bezier_df),
    "vae": to_scale_invariant(re_vae_df),
}
csgo_si_human = to_scale_invariant(csgo_games_df)
csgo_bots = {
    "stitch": to_scale_invariant(csgo_bots_stitch_df),
    "smooth": to_scale_invariant(csgo_bots_smooth_df),
    "bezier": to_scale_invariant(csgo_bots_bezier_df),
    "vae": to_scale_invariant(csgo_bots_vae_df),
}

ablations = {
    "EXT-10": EXT_FULL,
    "EXT-xy": EXT_NO_XY,
    "EXT-xy-vh": EXT_NO_XY_VH,
}

rows = []
for abl_name, cols in ablations.items():
    print(f"\n{'=' * 60}")
    print(f"=== Ablation {abl_name} ({len(cols)} feats) RE→CSGO ===")
    for bot in ["stitch", "smooth", "bezier", "vae"]:
        model, _ = train_bot_detector(
            re_si_human, re_bots[bot], cols,
            random_state=RNG_SEED, name=f"{abl_name} {bot}",
            show_feature_importance=False,
        )
        d = diagnose_cross_game(
            bot, model, csgo_si_human, csgo_bots[bot], cols,
            title_suffix=f"RE→CSGO {abl_name}",
            plot=False, plot_proba=False,
        )
        rows.append({
            "ablation": abl_name, "n_feats": len(cols), "bot": bot,
            "auc": d["auc"], "cal_detect": d["detect_cal"],
            "cal_fp": d["fp_cal"], "thr": d["thr"],
        })
        print()

summary = pd.DataFrame(rows)
print("\n=== RE→CSGO SI-EXT ablation summary ===")
print(
    summary.pivot_table(
        index="bot", columns="ablation", values=["auc", "cal_detect"],
    ).round(3).to_string()
)
print("\nVAE focus:")
print(summary[summary["bot"] == "vae"][
    ["ablation", "n_feats", "auc", "cal_detect", "cal_fp", "thr"]
].to_string(index=False))


## Feature scale comparison (RE vs CSGO)

In [ ]:
from src.features import cross_game_feature_cols

cols = cross_game_feature_cols

print("Feature medians (RE human vs CSGO human vs CSGO bots):")
compare_re_csgo = pd.DataFrame({
    "RE_human": re_games_df[cols].median(),
    "CSGO_human": csgo_games_df[cols].median(),
    "CSGO_stitch": csgo_bots_stitch_df[cols].median(),
    "CSGO_smooth": csgo_bots_smooth_df[cols].median(),
    "CSGO_bezier": csgo_bots_bezier_df[cols].median(),
    "CSGO_vae": csgo_bots_vae_df[cols].median(),
}).round(3)
print(compare_re_csgo)



## Cross-game transfer (CSGO train → RE test)

In [ ]:
from src.features import cross_game_feature_cols
from src.evaluation import train_bot_detector

print("=== CSGO model trained on STITCH bots (7 cross-game features) ===")
csgo_x_model_stitch, csgo_x_acc_stitch = train_bot_detector(
    csgo_games_df, csgo_bots_stitch_df, cross_game_feature_cols, name="CSGO stitch"
)
print()
print("=== CSGO model trained on SMOOTH bots (7 cross-game features) ===")
csgo_x_model_smooth, csgo_x_acc_smooth = train_bot_detector(
    csgo_games_df, csgo_bots_smooth_df, cross_game_feature_cols, name="CSGO smooth"
)
print()
print("=== CSGO model trained on BEZIER bots (7 cross-game features) ===")
csgo_x_model_bezier, csgo_x_acc_bezier = train_bot_detector(
    csgo_games_df, csgo_bots_bezier_df, cross_game_feature_cols, name="CSGO bezier"
)

print()
print("=== CSGO model trained on VAE bots (7 cross-game features) ===")
csgo_x_model_vae, csgo_x_acc_vae = train_bot_detector(
    csgo_games_df, csgo_bots_vae_df, cross_game_feature_cols, name="CSGO vae"
)


## Zero-shot diagnose (raw features, CSGO → RE)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import cross_game_feature_cols

print("=== Raw features: true zero-shot CSGO -> RE ===")
print("(threshold from RE humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", csgo_x_model_stitch, re_games_df, re_stitch_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "smooth", csgo_x_model_smooth, re_games_df, re_smooth_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "bezier", csgo_x_model_bezier, re_games_df, re_bezier_df,
    cross_game_feature_cols, title_suffix="raw",
)

print()
_ = diagnose_cross_game(
    "vae", csgo_x_model_vae, re_games_df, re_vae_df,
    cross_game_feature_cols, title_suffix="raw",
)


## Scale-invariant train (CSGO) — for RE transfer



In [ ]:
from src.evaluation import train_bot_detector
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS
from src.config import RNG_SEED

csgo_si_human_tr = to_scale_invariant(csgo_games_df)
csgo_si_stitch_tr = to_scale_invariant(csgo_bots_stitch_df)
csgo_si_smooth_tr = to_scale_invariant(csgo_bots_smooth_df)
csgo_si_bezier_tr = to_scale_invariant(csgo_bots_bezier_df)
csgo_si_vae_tr = to_scale_invariant(csgo_bots_vae_df)

print("=== Train on CSGO (scale-invariant features) ===")
m_csgo_si_stitch, _ = train_bot_detector(
    csgo_si_human_tr, csgo_si_stitch_tr, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="CSGO scale-invariant stitch",
    show_feature_importance=False,
)
print()
m_csgo_si_smooth, _ = train_bot_detector(
    csgo_si_human_tr, csgo_si_smooth_tr, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="CSGO scale-invariant smooth",
    show_feature_importance=False,
)
print()
m_csgo_si_bezier, _ = train_bot_detector(
    csgo_si_human_tr, csgo_si_bezier_tr, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="CSGO scale-invariant bezier",
    show_feature_importance=False,
)

print()
m_csgo_si_vae, _ = train_bot_detector(
    csgo_si_human_tr, csgo_si_vae_tr, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="CSGO scale-invariant vae",
    show_feature_importance=False,
)


## Scale-invariant EXT train (CSGO, 10 feats) — for RE transfer


In [ ]:
from src.evaluation import train_bot_detector
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS
from src.config import RNG_SEED

csgo_si_human_tr = to_scale_invariant(csgo_games_df)
csgo_si_stitch_tr = to_scale_invariant(csgo_bots_stitch_df)
csgo_si_smooth_tr = to_scale_invariant(csgo_bots_smooth_df)
csgo_si_bezier_tr = to_scale_invariant(csgo_bots_bezier_df)
csgo_si_vae_tr = to_scale_invariant(csgo_bots_vae_df)

print("=== Train on CSGO (scale-invariant EXT, 10 feats) ===")
print("cols:", SCALE_INVARIANT_EXT_COLS)
m_csgo_si_ext_stitch, _ = train_bot_detector(
    csgo_si_human_tr, csgo_si_stitch_tr, SCALE_INVARIANT_EXT_COLS,
    random_state=RNG_SEED, name="CSGO SI-EXT stitch",
    show_feature_importance=False,
)
print()
m_csgo_si_ext_smooth, _ = train_bot_detector(
    csgo_si_human_tr, csgo_si_smooth_tr, SCALE_INVARIANT_EXT_COLS,
    random_state=RNG_SEED, name="CSGO SI-EXT smooth",
    show_feature_importance=False,
)
print()
m_csgo_si_ext_bezier, _ = train_bot_detector(
    csgo_si_human_tr, csgo_si_bezier_tr, SCALE_INVARIANT_EXT_COLS,
    random_state=RNG_SEED, name="CSGO SI-EXT bezier",
    show_feature_importance=False,
)
print()
m_csgo_si_ext_vae, _ = train_bot_detector(
    csgo_si_human_tr, csgo_si_vae_tr, SCALE_INVARIANT_EXT_COLS,
    random_state=RNG_SEED, name="CSGO SI-EXT vae",
    show_feature_importance=False,
)


## Zero-shot diagnose (scale-invariant features, CSGO → RE)



In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS

re_si_human = to_scale_invariant(re_games_df)
re_si_stitch = to_scale_invariant(re_stitch_df)
re_si_smooth = to_scale_invariant(re_smooth_df)
re_si_bezier = to_scale_invariant(re_bezier_df)
re_si_vae = to_scale_invariant(re_vae_df)

print("=== Scale-invariant: true zero-shot CSGO -> RE ===")
print("(threshold from RE humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_csgo_si_stitch, re_si_human, re_si_stitch,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "smooth", m_csgo_si_smooth, re_si_human, re_si_smooth,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "bezier", m_csgo_si_bezier, re_si_human, re_si_bezier,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)

print()
_ = diagnose_cross_game(
    "vae", m_csgo_si_vae, re_si_human, re_si_vae,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)


## Zero-shot diagnose (scale-invariant EXT, CSGO → RE)


In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS

re_si_human = to_scale_invariant(re_games_df)
re_si_stitch = to_scale_invariant(re_stitch_df)
re_si_smooth = to_scale_invariant(re_smooth_df)
re_si_bezier = to_scale_invariant(re_bezier_df)
re_si_vae = to_scale_invariant(re_vae_df)

print("=== Scale-invariant EXT (10 feats): true zero-shot CSGO -> RE ===")
print("(threshold from RE humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_csgo_si_ext_stitch, re_si_human, re_si_stitch,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)
print()
_ = diagnose_cross_game(
    "smooth", m_csgo_si_ext_smooth, re_si_human, re_si_smooth,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)
print()
_ = diagnose_cross_game(
    "bezier", m_csgo_si_ext_bezier, re_si_human, re_si_bezier,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)
print()
_ = diagnose_cross_game(
    "vae", m_csgo_si_ext_vae, re_si_human, re_si_vae,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)


## SI-EXT ablation: drop `xy_corr` / `vh_ratio` (CSGO → RE)


In [ ]:
import pandas as pd
from src.evaluation import train_bot_detector, diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS
from src.config import RNG_SEED

EXT_FULL = list(SCALE_INVARIANT_EXT_COLS)
EXT_NO_XY = [c for c in EXT_FULL if c != "xy_corr"]
EXT_NO_XY_VH = [c for c in EXT_NO_XY if c != "vh_ratio"]

csgo_si_human_tr = to_scale_invariant(csgo_games_df)
csgo_bots_tr = {
    "stitch": to_scale_invariant(csgo_bots_stitch_df),
    "smooth": to_scale_invariant(csgo_bots_smooth_df),
    "bezier": to_scale_invariant(csgo_bots_bezier_df),
    "vae": to_scale_invariant(csgo_bots_vae_df),
}
re_si_human = to_scale_invariant(re_games_df)
re_bots = {
    "stitch": to_scale_invariant(re_stitch_df),
    "smooth": to_scale_invariant(re_smooth_df),
    "bezier": to_scale_invariant(re_bezier_df),
    "vae": to_scale_invariant(re_vae_df),
}

ablations = {
    "EXT-10": EXT_FULL,
    "EXT-xy": EXT_NO_XY,
    "EXT-xy-vh": EXT_NO_XY_VH,
}

rows = []
for abl_name, cols in ablations.items():
    print(f"\n{'=' * 60}")
    print(f"=== Ablation {abl_name} ({len(cols)} feats) CSGO→RE ===")
    for bot in ["stitch", "smooth", "bezier", "vae"]:
        model, _ = train_bot_detector(
            csgo_si_human_tr, csgo_bots_tr[bot], cols,
            random_state=RNG_SEED, name=f"CSGO {abl_name} {bot}",
            show_feature_importance=False,
        )
        d = diagnose_cross_game(
            bot, model, re_si_human, re_bots[bot], cols,
            title_suffix=f"CSGO→RE {abl_name}",
            plot=False, plot_proba=False,
        )
        rows.append({
            "ablation": abl_name, "n_feats": len(cols), "bot": bot,
            "auc": d["auc"], "cal_detect": d["detect_cal"],
            "cal_fp": d["fp_cal"], "thr": d["thr"],
        })
        print()

summary = pd.DataFrame(rows)
print("\n=== CSGO→RE SI-EXT ablation summary ===")
print(
    summary.pivot_table(
        index="bot", columns="ablation", values=["auc", "cal_detect"],
    ).round(3).to_string()
)
print("\nVAE focus:")
print(summary[summary["bot"] == "vae"][
    ["ablation", "n_feats", "auc", "cal_detect", "cal_fp", "thr"]
].to_string(index=False))


## Feature scale comparison (CSGO vs RE)

In [ ]:
from src.features import cross_game_feature_cols

cols = cross_game_feature_cols

print("Feature medians (CSGO human vs RE human vs RE bots):")
compare_csgo_re = pd.DataFrame({
    "CSGO_human": csgo_games_df[cols].median(),
    "RE_human": re_games_df[cols].median(),
    "RE_stitch": re_stitch_df[cols].median(),
    "RE_smooth": re_smooth_df[cols].median(),
    "RE_bezier": re_bezier_df[cols].median(),
    "RE_vae": re_vae_df[cols].median(),
}).round(3)
print(compare_csgo_re)

